In [ ]:
import numpy as np
import warnings
from scipy.optimize import minimize


class UnivariateMarkedHawkesMLE:
    """
    Hawkes univarié exponentiel avec variable explicative événementielle.

    Modèle :

        lambda(t) = mu
                    + sum_{t_k < t}
                      alpha * exp(eta * z_k)
                      * beta * exp(-beta * (t - t_k))

    où :

        z_k = standardisation de transform(mark_k)

    Paramètres estimés :

        mu    >= 0
        alpha >= 0
        eta   borné
        beta  > 0 si estimate_decay=True

    Interprétation :

        alpha : excitation moyenne de base
        beta  : vitesse de décroissance temporelle
        eta   : effet du mark sur l'amplitude du choc

    Si eta > 0 :
        les événements avec un mark élevé créent plus d'excitation future.

    Si eta < 0 :
        les événements avec un mark élevé créent moins d'excitation future.

    Remarque :
        ce modèle fait agir le mark sur l'amplitude du noyau, pas sur le decay.
    """

    def __init__(
        self,
        beta_init=1.0,
        estimate_decay=True,
        mark_transform="log1p",
        max_iter=3000,
        tol=1e-8,
        min_baseline=1e-12,
        min_decay=1e-8,
        alpha_upper=None,
        beta_upper=None,
        eta_bounds=(-5.0, 5.0),
        alpha_l2=0.0,
        beta_l2=0.0,
        eta_l2=0.0,
        n_starts=1,
        random_state=None,
    ):
        self.beta_init = float(beta_init)
        self.estimate_decay = bool(estimate_decay)
        self.mark_transform = mark_transform

        self.max_iter = int(max_iter)
        self.tol = float(tol)

        self.min_baseline = float(min_baseline)
        self.min_decay = float(min_decay)

        self.alpha_upper = alpha_upper
        self.beta_upper = beta_upper
        self.eta_bounds = eta_bounds

        self.alpha_l2 = float(alpha_l2)
        self.beta_l2 = float(beta_l2)
        self.eta_l2 = float(eta_l2)

        self.n_starts = int(n_starts)
        self.random_state = random_state

        if self.beta_init <= 0:
            raise ValueError("beta_init doit être strictement positif.")

    def _transform_mark_raw(self, marks):
        marks = np.asarray(marks, dtype=float).ravel()

        if self.mark_transform == "log1p":
            if np.any(marks <= -1):
                raise ValueError(
                    "Avec mark_transform='log1p', tous les marks doivent être > -1."
                )
            return np.log1p(marks)

        if self.mark_transform == "identity":
            return marks

        if callable(self.mark_transform):
            out = self.mark_transform(marks)
            return np.asarray(out, dtype=float).ravel()

        raise ValueError(
            "mark_transform doit être 'log1p', 'identity', ou une fonction callable."
        )

    @staticmethod
    def _prepare_one_realization(times, marks=None):
        times = np.asarray(times, dtype=float).ravel()

        if marks is None:
            marks = np.ones_like(times, dtype=float)
        else:
            marks = np.asarray(marks, dtype=float).ravel()

        if times.shape != marks.shape:
            raise ValueError("times et marks doivent avoir la même longueur.")

        if np.any(~np.isfinite(times)) or np.any(~np.isfinite(marks)):
            raise ValueError("times et marks doivent contenir des valeurs finies.")

        order = np.argsort(times)

        return times[order], marks[order]

    def _prepare_realizations(self, events, marks=None, end_times=None):
        """
        Formats acceptés.

        Une seule réalisation :

            events = np.array([t1, t2, ...])
            marks  = np.array([m1, m2, ...])

        Plusieurs jours / réalisations :

            events = [
                np.array([...]),  # jour 1
                np.array([...]),  # jour 2
                ...
            ]

            marks = [
                np.array([...]),  # marks jour 1
                np.array([...]),  # marks jour 2
                ...
            ]

        En univarié, chaque élément de events est une réalisation.
        """

        if isinstance(events, np.ndarray):
            t, m = self._prepare_one_realization(events, marks)
            realizations = [t]
            marks_realizations = [m]

        elif isinstance(events, (list, tuple)):
            if len(events) == 0:
                raise ValueError("events ne peut pas être vide.")

            # Liste de scalaires : une seule réalisation
            if all(np.ndim(x) == 0 for x in events):
                t, m = self._prepare_one_realization(events, marks)
                realizations = [t]
                marks_realizations = [m]

            # Liste d'arrays : plusieurs réalisations
            else:
                realizations = []
                marks_realizations = []

                if marks is None:
                    marks_iter = [None] * len(events)
                else:
                    if len(marks) != len(events):
                        raise ValueError(
                            "Pour plusieurs réalisations, marks doit avoir la même longueur que events."
                        )
                    marks_iter = marks

                for ev, mk in zip(events, marks_iter):
                    t, m = self._prepare_one_realization(ev, mk)
                    realizations.append(t)
                    marks_realizations.append(m)

        else:
            t, m = self._prepare_one_realization(events, marks)
            realizations = [t]
            marks_realizations = [m]

        if end_times is None:
            Ts = []

            for t in realizations:
                if len(t) == 0:
                    raise ValueError(
                        "end_times est requis si une réalisation ne contient aucun événement."
                    )

                Ts.append(float(t[-1]))

            warnings.warn(
                "end_times non fourni : utilisation du dernier timestamp de chaque réalisation. "
                "Pour une MLE correcte, fournissez l'horizon réel d'observation.",
                RuntimeWarning,
            )

        else:
            if np.ndim(end_times) == 0:
                Ts = [float(end_times)] * len(realizations)
            else:
                Ts = [float(x) for x in np.asarray(end_times, dtype=float).ravel()]

                if len(Ts) != len(realizations):
                    raise ValueError(
                        "end_times doit être un scalaire ou un array de longueur n_realizations."
                    )

        for idx, (t, T) in enumerate(zip(realizations, Ts)):
            if T <= 0 or not np.isfinite(T):
                raise ValueError("Chaque end_time doit être strictement positif et fini.")

            if np.any(t < 0):
                raise ValueError(f"Réalisation {idx}: timestamps négatifs.")

            if np.any(t > T):
                raise ValueError(
                    f"Réalisation {idx}: certains timestamps dépassent end_time."
                )

        return realizations, marks_realizations, np.asarray(Ts, dtype=float)

    def _fit_mark_standardization(self, marks_realizations):
        raw_all = []

        for marks in marks_realizations:
            raw = self._transform_mark_raw(marks)
            raw_all.append(raw)

        if len(raw_all) == 0:
            raise ValueError("Aucun mark disponible.")

        concatenated = np.concatenate(raw_all) if sum(len(x) for x in raw_all) > 0 else np.array([])

        if len(concatenated) == 0:
            mean = 0.0
            std = 1.0
        else:
            mean = float(np.mean(concatenated))
            std = float(np.std(concatenated))

            if std <= 1e-12:
                std = 1.0

        z_realizations = [(raw - mean) / std for raw in raw_all]

        return z_realizations, {"mean": mean, "std": std}

    def _transform_marks_with_stats(self, marks_realizations, stats):
        z_realizations = []

        for marks in marks_realizations:
            raw = self._transform_mark_raw(marks)
            z = (raw - stats["mean"]) / stats["std"]
            z_realizations.append(z)

        return z_realizations

    def _unpack(self, theta):
        if self.estimate_decay:
            mu, alpha, beta, eta = theta
        else:
            mu, alpha, eta = theta
            beta = self.beta_init

        return float(mu), float(alpha), float(beta), float(eta)

    def _kernel_integral_and_grads(self, times, z, T, beta, eta):
        """
        Compensateur du noyau marqué :

            S = sum_k w_k * (1 - exp(-beta * (T - t_k)))

        avec :

            w_k = exp(eta * z_k)

        Dérivées :

            dS/deta = sum_k w_k z_k (1 - exp(-beta * rem_k))

            dS/dbeta = sum_k w_k rem_k exp(-beta * rem_k)
        """
        if len(times) == 0:
            return 0.0, 0.0, 0.0

        rem = T - times
        w = np.exp(eta * z)
        e = np.exp(-beta * rem)

        S = np.sum(w * (1.0 - e))
        S_eta = np.sum(w * z * (1.0 - e))
        S_beta = np.sum(w * rem * e)

        return float(S), float(S_eta), float(S_beta)

    def _nll_grad_one(self, theta, times, z, T):
        mu, alpha, beta, eta = self._unpack(theta)

        if mu <= 0 or alpha < 0 or beta <= 0:
            return np.inf, np.zeros_like(theta)

        ll = 0.0

        grad_mu = 0.0
        grad_alpha = 0.0
        grad_beta = 0.0
        grad_eta = 0.0

        # r(t) = beta * sum_{t_k<t} exp(eta z_k) exp(-beta(t-t_k))
        #
        # q(t) = dr(t)/dbeta
        #
        # u(t) = dr(t)/deta
        r = 0.0
        q = 0.0
        u = 0.0

        last_t = 0.0
        n = len(times)
        k = 0

        while k < n:
            t = times[k]
            dt = t - last_t

            if dt < -1e-12:
                raise RuntimeError("Les timestamps doivent être triés.")

            if dt > 0:
                e = np.exp(-beta * dt)

                # q doit utiliser l'ancien r.
                q = e * (q - dt * r)
                r = e * r
                u = e * u

            # Groupe de timestamps égaux.
            # Les événements simultanés ne s'excitent pas entre eux.
            k2 = k + 1
            while k2 < n and times[k2] == t:
                k2 += 1

            z_group = z[k:k2]
            count = k2 - k

            lam = mu + alpha * r

            if lam <= 0 or not np.isfinite(lam):
                return np.inf, np.zeros_like(theta)

            ll += count * np.log(lam)

            inv_lam = 1.0 / lam

            grad_mu += count * inv_lam
            grad_alpha += count * r * inv_lam

            if self.estimate_decay:
                grad_beta += count * alpha * q * inv_lam

            grad_eta += count * alpha * u * inv_lam

            # Sauts après évaluation de lambda(t)
            w_group = np.exp(eta * z_group)

            w_sum = np.sum(w_group)
            wz_sum = np.sum(w_group * z_group)

            r += beta * w_sum

            if self.estimate_decay:
                # dérivée du saut beta * w par rapport à beta
                q += w_sum

            # dérivée du saut beta * w par rapport à eta
            u += beta * wz_sum

            last_t = t
            k = k2

        S, S_eta, S_beta = self._kernel_integral_and_grads(
            times=times,
            z=z,
            T=T,
            beta=beta,
            eta=eta,
        )

        # Compensateur :
        #
        # int_0^T lambda(s) ds = mu T + alpha S
        ll -= mu * T + alpha * S

        grad_mu -= T
        grad_alpha -= S

        if self.estimate_decay:
            grad_beta -= alpha * S_beta

        grad_eta -= alpha * S_eta

        nll = -ll

        grad_mu = -grad_mu
        grad_alpha = -grad_alpha
        grad_beta = -grad_beta
        grad_eta = -grad_eta

        if self.alpha_l2 > 0:
            nll += 0.5 * self.alpha_l2 * alpha * alpha
            grad_alpha += self.alpha_l2 * alpha

        if self.estimate_decay and self.beta_l2 > 0:
            nll += 0.5 * self.beta_l2 * beta * beta
            grad_beta += self.beta_l2 * beta

        if self.eta_l2 > 0:
            nll += 0.5 * self.eta_l2 * eta * eta
            grad_eta += self.eta_l2 * eta

        if self.estimate_decay:
            grad = np.array([grad_mu, grad_alpha, grad_beta, grad_eta], dtype=float)
        else:
            grad = np.array([grad_mu, grad_alpha, grad_eta], dtype=float)

        return float(nll), grad

    def _nll_grad_all(self, theta, realizations, z_realizations, end_times):
        nll_total = 0.0
        grad_total = np.zeros_like(theta, dtype=float)

        for times, z, T in zip(realizations, z_realizations, end_times):
            nll, grad = self._nll_grad_one(theta, times, z, float(T))

            if not np.isfinite(nll):
                return np.inf, np.zeros_like(theta)

            nll_total += nll
            grad_total += grad

        return float(nll_total), grad_total

    def _initial_theta(self, realizations, end_times):
        total_events = sum(len(x) for x in realizations)
        total_T = float(np.sum(end_times))

        mu0 = max(0.5 * total_events / max(total_T, 1e-12), self.min_baseline * 10)
        alpha0 = 0.05
        eta0 = 0.0

        if self.estimate_decay:
            beta0 = max(self.beta_init, self.min_decay * 10)

            if self.beta_upper is not None:
                beta0 = min(beta0, self.beta_upper * 0.9)

            return np.array([mu0, alpha0, beta0, eta0], dtype=float)

        return np.array([mu0, alpha0, eta0], dtype=float)

    def fit(self, events, marks=None, end_times=None, x0=None):
        realizations, marks_realizations, end_times = self._prepare_realizations(
            events=events,
            marks=marks,
            end_times=end_times,
        )

        z_realizations, mark_stats = self._fit_mark_standardization(marks_realizations)

        if x0 is None:
            theta0 = self._initial_theta(realizations, end_times)
        else:
            theta0 = np.asarray(x0, dtype=float).ravel()

            expected = 4 if self.estimate_decay else 3

            if theta0.size != expected:
                raise ValueError(f"x0 doit avoir une longueur {expected}.")

        if self.estimate_decay:
            bounds = [
                (self.min_baseline, None),
                (0.0, self.alpha_upper),
                (self.min_decay, self.beta_upper),
                self.eta_bounds,
            ]
        else:
            bounds = [
                (self.min_baseline, None),
                (0.0, self.alpha_upper),
                self.eta_bounds,
            ]

        rng = np.random.default_rng(self.random_state)

        best_result = None
        best_fun = np.inf

        for start in range(max(1, self.n_starts)):
            if start == 0:
                start_theta = theta0.copy()
            else:
                if self.estimate_decay:
                    mu0, alpha0, beta0, eta0 = theta0

                    start_theta = np.array(
                        [
                            mu0 * rng.lognormal(0.0, 0.4),
                            alpha0 * rng.lognormal(0.0, 0.7),
                            beta0 * rng.lognormal(0.0, 0.7),
                            eta0 + rng.normal(0.0, 0.4),
                        ],
                        dtype=float,
                    )
                else:
                    mu0, alpha0, eta0 = theta0

                    start_theta = np.array(
                        [
                            mu0 * rng.lognormal(0.0, 0.4),
                            alpha0 * rng.lognormal(0.0, 0.7),
                            eta0 + rng.normal(0.0, 0.4),
                        ],
                        dtype=float,
                    )

            # Projection de l'initialisation dans les bornes
            for idx, (lo, hi) in enumerate(bounds):
                if lo is not None and start_theta[idx] < lo:
                    start_theta[idx] = lo * 10.0 if lo > 0 else lo

                if hi is not None and start_theta[idx] > hi:
                    start_theta[idx] = hi * 0.9 if hi > 0 else hi

            result = minimize(
                fun=lambda th: self._nll_grad_all(
                    th,
                    realizations,
                    z_realizations,
                    end_times,
                ),
                x0=start_theta,
                jac=True,
                bounds=bounds,
                method="L-BFGS-B",
                options={
                    "maxiter": self.max_iter,
                    "ftol": self.tol,
                },
            )

            if result.fun < best_fun:
                best_fun = float(result.fun)
                best_result = result

        self.result_ = best_result
        self.success_ = bool(best_result.success)
        self.message_ = best_result.message
        self.n_iter_ = best_result.nit

        self.events_ = realizations
        self.marks_ = marks_realizations
        self.z_marks_ = z_realizations
        self.end_times_ = end_times
        self.mark_stats_ = mark_stats

        self.baseline_, self.alpha_, self.beta_, self.eta_ = self._unpack(best_result.x)

        self.log_likelihood_ = -float(best_result.fun)

        # Matrice de branchement effective univariée :
        #
        # B = alpha * E[exp(eta Z)]
        all_z = np.concatenate(z_realizations) if sum(len(z) for z in z_realizations) > 0 else np.array([])

        if len(all_z) > 0:
            self.mean_mark_weight_ = float(np.mean(np.exp(self.eta_ * all_z)))
        else:
            self.mean_mark_weight_ = 1.0

        self.branching_ratio_ = float(self.alpha_ * self.mean_mark_weight_)
        self.is_stable_ = bool(self.branching_ratio_ < 1.0)

        return self

    def score(self, events=None, marks=None, end_times=None):
        if not hasattr(self, "baseline_"):
            raise RuntimeError("Le modèle doit être fitté avant score().")

        if events is None:
            realizations = self.events_
            z_realizations = self.z_marks_
            end_times = self.end_times_
        else:
            realizations, marks_realizations, end_times = self._prepare_realizations(
                events=events,
                marks=marks,
                end_times=end_times,
            )

            z_realizations = self._transform_marks_with_stats(
                marks_realizations,
                self.mark_stats_,
            )

        if self.estimate_decay:
            theta = np.array(
                [
                    self.baseline_,
                    self.alpha_,
                    self.beta_,
                    self.eta_,
                ],
                dtype=float,
            )
        else:
            theta = np.array(
                [
                    self.baseline_,
                    self.alpha_,
                    self.eta_,
                ],
                dtype=float,
            )

        nll, _ = self._nll_grad_all(
            theta,
            realizations,
            z_realizations,
            end_times,
        )

        return -float(nll)

    def intensity_at_events(self, events=None, marks=None):
        """
        Retourne lambda(t_k-) aux temps d'événements.

        Utile pour diagnostiquer si le modèle donne une forte intensité
        aux timestamps observés.

        Ne supporte ici qu'une seule réalisation pour rester simple.
        """
        if not hasattr(self, "baseline_"):
            raise RuntimeError("Le modèle doit être fitté avant intensity_at_events().")

        if events is None:
            times = self.events_[0]
            z = self.z_marks_[0]
        else:
            realizations, marks_realizations, _ = self._prepare_realizations(
                events=events,
                marks=marks,
                end_times=float(np.max(events)) if len(events) else 1.0,
            )

            if len(realizations) != 1:
                raise ValueError("intensity_at_events attend une seule réalisation.")

            times = realizations[0]
            z = self._transform_marks_with_stats(
                marks_realizations,
                self.mark_stats_,
            )[0]

        mu = self.baseline_
        alpha = self.alpha_
        beta = self.beta_
        eta = self.eta_

        intensities = np.zeros(len(times), dtype=float)

        r = 0.0
        last_t = 0.0

        n = len(times)
        k = 0

        while k < n:
            t = times[k]
            dt = t - last_t

            if dt > 0:
                r *= np.exp(-beta * dt)

            k2 = k + 1
            while k2 < n and times[k2] == t:
                k2 += 1

            lam = mu + alpha * r
            intensities[k:k2] = lam

            z_group = z[k:k2]
            w_group = np.exp(eta * z_group)

            r += beta * np.sum(w_group)

            last_t = t
            k = k2

        return intensities

    def get_params(self):
        if not hasattr(self, "baseline_"):
            raise RuntimeError("Le modèle doit être fitté avant get_params().")

        return {
            "baseline": self.baseline_,
            "alpha": self.alpha_,
            "beta": self.beta_,
            "eta": self.eta_,
            "mark_transform": self.mark_transform,
            "mark_stats": dict(self.mark_stats_),
            "mean_mark_weight": self.mean_mark_weight_,
            "branching_ratio": self.branching_ratio_,
            "is_stable": self.is_stable_,
            "log_likelihood": self.log_likelihood_,
            "success": self.success_,
            "message": self.message_,
            "n_iter": self.n_iter_,
        }

In [ ]:
import numpy as np

times = np.array([0.10, 0.30, 0.55, 1.20, 1.70, 2.10, 2.80])
volume = np.array([100, 250, 80, 1000, 300, 150, 700])

model = UnivariateMarkedHawkesMLE(
    beta_init=1.0,
    estimate_decay=True,
    mark_transform="log1p",
    alpha_upper=0.99,
    beta_upper=20.0,
    eta_bounds=(-3.0, 3.0),
    eta_l2=1e-4,
    n_starts=10,
    random_state=42,
)

model.fit(
    events=times,
    marks=volume,
    end_times=3.0,
)

print(model.get_params())

In [ ]:
events = [
    np.array([0.10, 0.40, 0.90, 1.50]),
    np.array([0.20, 0.80, 1.10, 2.00, 2.30]),
    np.array([0.15, 0.70, 1.40]),
]

marks = [
    np.array([100, 300, 150, 800]),
    np.array([200, 100, 500, 900, 250]),
    np.array([120, 600, 300]),
]

end_times = np.array([2.0, 2.5, 1.8])

model = UnivariateMarkedHawkesMLE(
    beta_init=1.0,
    estimate_decay=True,
    beta_upper=20.0,
    alpha_upper=0.99,
    eta_bounds=(-3.0, 3.0),
    n_starts=10,
    random_state=123,
)

model.fit(
    events=events,
    marks=marks,
    end_times=end_times,
)

params = model.get_params()

print("mu:", params["baseline"])
print("alpha:", params["alpha"])
print("beta:", params["beta"])
print("eta:", params["eta"])
print("branching ratio:", params["branching_ratio"])
print("stable:", params["is_stable"])

In [ ]:
# like estimators
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import numpy as np
from numpy.typing import ArrayLike
from scipy.optimize import minimize


def _as_sorted_1d_times(times: ArrayLike) -> np.ndarray:
    """Convert input to a sorted one-dimensional array of event times."""
    times = np.asarray(times, dtype=float).ravel()

    if times.size == 0:
        raise ValueError("times must contain at least one event.")

    if np.any(~np.isfinite(times)):
        raise ValueError("times must contain finite values only.")

    if np.any(times < 0):
        raise ValueError("times must be non-negative.")

    return np.sort(times)


def _as_1d_marks(marks: ArrayLike, n: int) -> np.ndarray:
    """Convert marks to a one-dimensional array aligned with event times."""
    marks = np.asarray(marks, dtype=float).ravel()

    if marks.size != n:
        raise ValueError("marks must have the same length as times.")

    if np.any(~np.isfinite(marks)):
        raise ValueError("marks must contain finite values only.")

    return marks


def _sort_times_and_marks(times: ArrayLike, marks: ArrayLike) -> Tuple[np.ndarray, np.ndarray]:
    """Sort event times and align marks with the sorted event times."""
    times = np.asarray(times, dtype=float).ravel()
    marks = np.asarray(marks, dtype=float).ravel()

    if times.size == 0:
        raise ValueError("times must contain at least one event.")

    if marks.size != times.size:
        raise ValueError("marks must have the same length as times.")

    if np.any(~np.isfinite(times)) or np.any(~np.isfinite(marks)):
        raise ValueError("times and marks must contain finite values only.")

    if np.any(times < 0):
        raise ValueError("times must be non-negative.")

    order = np.argsort(times)
    return times[order], marks[order]


@dataclass
class UnivariateMarkedHawkesExpMLE:
    """MLE for a univariate marked exponential Hawkes process.

    Model convention
    ----------------
    lambda(t) = mu + sum_{tj < t} alpha * w_j(eta) * exp(-beta * (t - tj))

    where:

        w_j(eta) = exp(eta * z_j)

    and z_j is a standardized transformation of mark_j.

    By default:

        z_j = standardize(log(1 + mark_j))

    Normalized-mark convention
    --------------------------
    If normalize_mark_weight=True, the code uses:

        w_j(eta) = exp(eta * z_j) / mean_k exp(eta * z_k)

    on the estimation sample.

    This keeps the empirical average mark weight equal to 1, so that alpha / beta
    remains interpretable as an average branching ratio.

    Stability condition
    -------------------
    Without marks:

        integral_0^infty alpha exp(-beta t) dt = alpha / beta < 1.

    With marks:

        branching ratio = (alpha / beta) * E[w(M)].

    If normalize_mark_weight=True, E[w(M)] is approximately 1 on the training
    sample, so the stationarity condition is again alpha / beta < 1.

    Parameters estimated
    --------------------
    mu > 0, alpha >= 0, beta > 0, eta real.

    If enforce_stationarity=True and normalize_mark_weight=True, the
    reparameterization enforces alpha < beta.

    Interpretation
    --------------
    eta > 0:
        high marks increase future excitation.

    eta < 0:
        high marks reduce future excitation.

    eta = 0:
        the model reduces to the unmarked Hawkes model.

    Implementation details
    ----------------------
    The exponential kernel permits an O(n) likelihood recursion:

        A_i = sum_{j<i} w_j exp(-beta * (t_i - t_j))

        A_i = exp(-beta * (t_i - t_{i-1})) * (w_{i-1} + A_{i-1})

    where w_j is the mark weight of event j.
    """

    T: Optional[float] = None
    enforce_stationarity: bool = True
    normalize_mark_weight: bool = True
    mark_transform: str = "log1p"
    epsilon: float = 1e-9

    result_: Optional[object] = None
    params_: Optional[Dict[str, float]] = None
    loglik_: Optional[float] = None
    mark_stats_: Optional[Dict[str, float]] = None

    @staticmethod
    def _sigmoid(x: float) -> float:
        """Numerically stable sigmoid."""
        if x >= 0:
            z = np.exp(-x)
            return float(1.0 / (1.0 + z))
        z = np.exp(x)
        return float(z / (1.0 + z))

    @staticmethod
    def _logit(p: float) -> float:
        """Logit transform with clipping."""
        p = np.clip(p, 1e-6, 1.0 - 1e-6)
        return float(np.log(p / (1.0 - p)))

    def _transform_marks(self, marks: np.ndarray) -> np.ndarray:
        """Apply the raw mark transformation before standardization."""
        marks = np.asarray(marks, dtype=float).ravel()

        if self.mark_transform == "log1p":
            if np.any(marks <= -1):
                raise ValueError("With mark_transform='log1p', marks must be > -1.")
            return np.log1p(marks)

        if self.mark_transform == "identity":
            return marks

        raise ValueError("mark_transform must be either 'log1p' or 'identity'.")

    def _fit_mark_standardization(self, marks: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
        """Fit mark standardization on the training sample."""
        x = self._transform_marks(marks)

        mean = float(np.mean(x))
        std = float(np.std(x))

        if std <= self.epsilon:
            std = 1.0

        z = (x - mean) / std

        stats = {
            "mean": mean,
            "std": std,
        }

        return z, stats

    def _apply_mark_standardization(self, marks: np.ndarray) -> np.ndarray:
        """Apply fitted mark standardization to new marks."""
        if self.mark_stats_ is None:
            raise RuntimeError("fit must be called before standardizing new marks.")

        x = self._transform_marks(marks)
        return (x - self.mark_stats_["mean"]) / self.mark_stats_["std"]

    @staticmethod
    def _mark_weights(
        z: np.ndarray,
        eta: float,
        normalize: bool = True,
        epsilon: float = 1e-12,
    ) -> np.ndarray:
        """Compute w_j(eta) = exp(eta z_j), optionally normalized to mean 1."""
        raw = np.exp(eta * z)

        if normalize:
            denom = float(np.mean(raw))
            denom = max(denom, epsilon)
            return raw / denom

        return raw

    @staticmethod
    def _recursive_A(times: np.ndarray, weights: np.ndarray, beta: float) -> np.ndarray:
        """Compute A_i = sum_{j<i} w_j exp(-beta (t_i - t_j)) in O(n)."""
        n = len(times)
        A = np.zeros(n, dtype=float)

        for i in range(1, n):
            dt = times[i] - times[i - 1]
            A[i] = np.exp(-beta * dt) * (weights[i - 1] + A[i - 1])

        return A

    @classmethod
    def loglikelihood(
        cls,
        times: ArrayLike,
        marks: ArrayLike,
        T: float,
        mu: float,
        alpha: float,
        beta: float,
        eta: float,
        mark_mean: Optional[float] = None,
        mark_std: Optional[float] = None,
        mark_transform: str = "log1p",
        normalize_mark_weight: bool = True,
        enforce_stationarity: bool = True,
        epsilon: float = 1e-9,
    ) -> float:
        """Evaluate the marked Hawkes log-likelihood for given parameters."""
        times, marks = _sort_times_and_marks(times, marks)

        if T < times[-1]:
            raise ValueError("T must be >= last event time.")

        if mu <= 0 or alpha < 0 or beta <= 0:
            return -np.inf

        if mark_transform == "log1p":
            if np.any(marks <= -1):
                raise ValueError("With mark_transform='log1p', marks must be > -1.")
            x = np.log1p(marks)
        elif mark_transform == "identity":
            x = marks
        else:
            raise ValueError("mark_transform must be either 'log1p' or 'identity'.")

        if mark_mean is None:
            mark_mean = float(np.mean(x))

        if mark_std is None:
            mark_std = float(np.std(x))
            if mark_std <= epsilon:
                mark_std = 1.0

        z = (x - mark_mean) / mark_std

        weights = cls._mark_weights(
            z=z,
            eta=eta,
            normalize=normalize_mark_weight,
            epsilon=epsilon,
        )

        if enforce_stationarity:
            empirical_branching = (alpha / beta) * float(np.mean(weights))
            if empirical_branching >= 1.0:
                return -np.inf

        A = cls._recursive_A(times, weights, beta)
        intensities = mu + alpha * A

        if np.any(intensities <= 0) or np.any(~np.isfinite(intensities)):
            return -np.inf

        compensator = (
            mu * T
            + (alpha / beta)
            * np.sum(weights * (1.0 - np.exp(-beta * (T - times))))
        )

        return float(np.sum(np.log(intensities)) - compensator)

    def fit(
        self,
        times: ArrayLike,
        marks: ArrayLike,
        initial: Optional[Tuple[float, float, float, float]] = None,
    ) -> "UnivariateMarkedHawkesExpMLE":
        """Fit the marked Hawkes model by numerical minimization of the negative log-likelihood."""
        times, marks = _sort_times_and_marks(times, marks)

        T = float(times[-1] if self.T is None else self.T)

        if T < times[-1]:
            raise ValueError("T must be >= last event time.")

        n = len(times)

        z, mark_stats = self._fit_mark_standardization(marks)
        self.mark_stats_ = mark_stats

        if initial is None:
            empirical_rate = n / T

            mu0 = max(0.5 * empirical_rate, self.epsilon)

            inter_times = np.diff(np.r_[0.0, times])
            median_duration = max(float(np.median(inter_times)), self.epsilon)

            beta0 = 1.0 / median_duration
            alpha0 = 0.5 * beta0
            eta0 = 0.0

            initial = (mu0, alpha0, beta0, eta0)

        mu0, alpha0, beta0, eta0 = map(float, initial)

        if mu0 <= 0 or alpha0 < 0 or beta0 <= 0:
            raise ValueError("initial must satisfy mu>0, alpha>=0, beta>0.")

        if self.enforce_stationarity and self.normalize_mark_weight:
            if alpha0 >= beta0:
                alpha0 = 0.5 * beta0

        # Reparameterization:
        #
        # mu  = exp(x0)
        # beta = exp(x1)
        #
        # If stationarity is enforced and mark weights are normalized:
        #
        # alpha = beta * sigmoid(x2)
        #
        # which enforces alpha / beta in (0, 1).
        #
        # eta = x3
        x0 = np.array(
            [
                np.log(mu0),
                np.log(beta0),
                self._logit(alpha0 / beta0 if beta0 > 0 else 0.5)
                if self.enforce_stationarity and self.normalize_mark_weight
                else np.log(max(alpha0, self.epsilon)),
                eta0,
            ],
            dtype=float,
        )

        def unpack(x: np.ndarray) -> Tuple[float, float, float, float]:
            mu = float(np.exp(x[0]))
            beta = float(np.exp(x[1]))

            if self.enforce_stationarity and self.normalize_mark_weight:
                ratio = self._sigmoid(float(x[2]))
                alpha = beta * ratio
            else:
                alpha = float(np.exp(x[2]))

            eta = float(x[3])

            return mu, alpha, beta, eta

        def objective(x: np.ndarray) -> float:
            mu, alpha, beta, eta = unpack(x)

            ll = self.loglikelihood(
                times=times,
                marks=marks,
                T=T,
                mu=mu,
                alpha=alpha,
                beta=beta,
                eta=eta,
                mark_mean=self.mark_stats_["mean"],
                mark_std=self.mark_stats_["std"],
                mark_transform=self.mark_transform,
                normalize_mark_weight=self.normalize_mark_weight,
                enforce_stationarity=self.enforce_stationarity,
                epsilon=self.epsilon,
            )

            return -ll if np.isfinite(ll) else 1e100

        res = minimize(
            objective,
            x0,
            method="L-BFGS-B",
        )

        mu, alpha, beta, eta = unpack(res.x)

        weights = self._mark_weights(
            z=z,
            eta=eta,
            normalize=self.normalize_mark_weight,
            epsilon=self.epsilon,
        )

        mean_mark_weight = float(np.mean(weights))
        branching_ratio = (alpha / beta) * mean_mark_weight

        self.result_ = res

        self.params_ = {
            "mu": mu,
            "alpha": alpha,
            "beta": beta,
            "eta": eta,
            "mean_mark_weight": mean_mark_weight,
            "branching_ratio": branching_ratio,
            "mark_mean": self.mark_stats_["mean"],
            "mark_std": self.mark_stats_["std"],
        }

        self.loglik_ = self.loglikelihood(
            times=times,
            marks=marks,
            T=T,
            mu=mu,
            alpha=alpha,
            beta=beta,
            eta=eta,
            mark_mean=self.mark_stats_["mean"],
            mark_std=self.mark_stats_["std"],
            mark_transform=self.mark_transform,
            normalize_mark_weight=self.normalize_mark_weight,
            enforce_stationarity=self.enforce_stationarity,
            epsilon=self.epsilon,
        )

        return self

    def intensity_at_events(self, times: ArrayLike, marks: ArrayLike) -> np.ndarray:
        """Return fitted intensities evaluated at observed event times."""
        if self.params_ is None:
            raise RuntimeError("fit must be called before intensity_at_events.")

        times, marks = _sort_times_and_marks(times, marks)

        z = self._apply_mark_standardization(marks)

        weights = self._mark_weights(
            z=z,
            eta=self.params_["eta"],
            normalize=self.normalize_mark_weight,
            epsilon=self.epsilon,
        )

        A = self._recursive_A(
            times=times,
            weights=weights,
            beta=self.params_["beta"],
        )

        return self.params_["mu"] + self.params_["alpha"] * A

    def mark_weights(self, marks: ArrayLike) -> np.ndarray:
        """Return fitted mark weights w_j(eta) for a vector of marks."""
        if self.params_ is None:
            raise RuntimeError("fit must be called before mark_weights.")

        marks = np.asarray(marks, dtype=float).ravel()
        z = self._apply_mark_standardization(marks)

        return self._mark_weights(
            z=z,
            eta=self.params_["eta"],
            normalize=self.normalize_mark_weight,
            epsilon=self.epsilon,
        )
        
import numpy as np

times = np.array([0.10, 0.30, 0.55, 1.20, 1.70, 2.10, 2.80])
marks = np.array([100, 250, 80, 1000, 300, 150, 700])

model = UnivariateMarkedHawkesExpMLE(
    T=3.0,
    enforce_stationarity=True,
    normalize_mark_weight=True,
    mark_transform="log1p",
)

model.fit(times, marks)

print(model.params_)
print("loglik:", model.loglik_)

lambdas = model.intensity_at_events(times, marks)
print(lambdas)

weights = model.mark_weights(marks)
print(weights)

In [ ]:
# Compensateur 

def _fitted_mark_weights_from_z(self, z: np.ndarray) -> np.ndarray:
    """Compute fitted mark weights using the training normalization."""
    if self.params_ is None:
        raise RuntimeError("fit must be called before computing fitted mark weights.")

    eta = self.params_["eta"]
    raw = np.exp(eta * z)

    if self.normalize_mark_weight:
        if self.mark_weight_norm_ is None:
            raise RuntimeError("mark_weight_norm_ is missing. Refit the model.")
        return raw / max(self.mark_weight_norm_, self.epsilon)

    return raw


def cumulative_intensity(
    self,
    times: ArrayLike,
    marks: ArrayLike,
    eval_times: Optional[ArrayLike] = None,
) -> np.ndarray:
    """
    Compute the fitted cumulative intensity Lambda(t).

    Model
    -----
    lambda(t) = mu + sum_{tj < t} alpha * w_j * exp(-beta * (t - tj))

    Cumulative intensity
    --------------------
    Lambda(t) = int_0^t lambda(s) ds

              = mu * t
                + (alpha / beta)
                  * sum_{tj <= t} w_j * (1 - exp(-beta * (t - tj)))

    Parameters
    ----------
    times:
        Event times.

    marks:
        Event marks aligned with times.

    eval_times:
        Times where Lambda(t) is evaluated.
        If None, evaluates Lambda at observed event times.

    Returns
    -------
    np.ndarray
        Values Lambda(eval_times).
    """
    if self.params_ is None:
        raise RuntimeError("fit must be called before cumulative_intensity.")

    times, marks = _sort_times_and_marks(times, marks)

    if eval_times is None:
        eval_times_arr = times.copy()
    else:
        eval_times_arr = np.asarray(eval_times, dtype=float).ravel()

    if eval_times_arr.size == 0:
        return np.array([], dtype=float)

    if np.any(~np.isfinite(eval_times_arr)):
        raise ValueError("eval_times must contain finite values only.")

    if np.any(eval_times_arr < 0):
        raise ValueError("eval_times must be non-negative.")

    z = self._apply_mark_standardization(marks)
    weights = self._fitted_mark_weights_from_z(z)

    mu = self.params_["mu"]
    alpha = self.params_["alpha"]
    beta = self.params_["beta"]

    order = np.argsort(eval_times_arr)
    eval_sorted = eval_times_arr[order]

    Lambda_sorted = np.zeros_like(eval_sorted, dtype=float)

    # We maintain:
    #
    # sum_w(t)   = sum_{tj <= t} w_j
    # decay_w(t) = sum_{tj <= t} w_j exp(-beta * (t - tj))
    #
    # Then:
    #
    # Lambda(t) = mu * t + (alpha / beta) * (sum_w(t) - decay_w(t))
    event_idx = 0
    n_events = len(times)

    current_time = 0.0
    sum_w = 0.0
    decay_w = 0.0

    for r, t_eval in enumerate(eval_sorted):
        # Process all events up to t_eval.
        while event_idx < n_events and times[event_idx] <= t_eval:
            t_event = times[event_idx]

            dt = t_event - current_time

            if dt < -1e-12:
                raise RuntimeError("Internal ordering error.")

            if dt > 0:
                decay_w *= np.exp(-beta * dt)
                current_time = t_event

            # Group simultaneous events.
            j = event_idx
            w_sum = 0.0

            while j < n_events and times[j] == t_event:
                w_sum += weights[j]
                j += 1

            sum_w += w_sum
            decay_w += w_sum

            event_idx = j

        # Move from current_time to t_eval.
        dt = t_eval - current_time

        if dt < -1e-12:
            raise RuntimeError("eval_times must be processed in sorted order.")

        if dt > 0:
            decay_w *= np.exp(-beta * dt)
            current_time = t_eval

        Lambda_sorted[r] = mu * t_eval + (alpha / beta) * (sum_w - decay_w)

    # Return to original eval_times order.
    Lambda = np.empty_like(Lambda_sorted)
    Lambda[order] = Lambda_sorted

    return Lambda


def compensator(
    self,
    times: ArrayLike,
    marks: ArrayLike,
    T: Optional[float] = None,
) -> float:
    """
    Compute the compensator over [0, T]:

        Lambda(T) = int_0^T lambda(s) ds

    This is the integral term used in the log-likelihood.
    """
    if self.params_ is None:
        raise RuntimeError("fit must be called before compensator.")

    times, marks = _sort_times_and_marks(times, marks)

    if T is None:
        if self.T is not None:
            T_eval = float(self.T)
        else:
            T_eval = float(times[-1])
    else:
        T_eval = float(T)

    if T_eval < times[-1]:
        raise ValueError("T must be >= last event time.")

    return float(
        self.cumulative_intensity(
            times=times,
            marks=marks,
            eval_times=np.array([T_eval]),
        )[0]
    )


def cumulative_intensity_at_events(
    self,
    times: ArrayLike,
    marks: ArrayLike,
) -> np.ndarray:
    """
    Compute Lambda(t_i) at observed event times.

    Useful for time-rescaling diagnostics.
    """
    times, marks = _sort_times_and_marks(times, marks)

    return self.cumulative_intensity(
        times=times,
        marks=marks,
        eval_times=times,
    )


def rescaled_interarrival_times(
    self,
    times: ArrayLike,
    marks: ArrayLike,
) -> np.ndarray:
    """
    Compute time-rescaled interarrival times.

    If the model is correctly specified, the values

        Z_i = Lambda(t_i) - Lambda(t_{i-1})

    should approximately follow Exp(1), with Lambda(t_0)=0.
    """
    Lambda_events = self.cumulative_intensity_at_events(times, marks)

    return np.diff(
        np.r_[0.0, Lambda_events]
    )


def transformed_uniform_residuals(
    self,
    times: ArrayLike,
    marks: ArrayLike,
) -> np.ndarray:
    """
    Compute uniform residuals from time-rescaled interarrival times.

    If the model is correctly specified:

        Z_i ~ Exp(1)

    so:

        U_i = 1 - exp(-Z_i) ~ Uniform(0, 1)
    """
    z = self.rescaled_interarrival_times(times, marks)

    return 1.0 - np.exp(-z)

In [ ]:
# time scaled

from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import numpy as np
from numpy.typing import ArrayLike
from scipy.optimize import minimize


def _sort_times_and_marks(times: ArrayLike, marks: ArrayLike) -> Tuple[np.ndarray, np.ndarray]:
    """Sort event times and align marks."""
    times = np.asarray(times, dtype=float).ravel()
    marks = np.asarray(marks, dtype=float).ravel()

    if times.size == 0:
        raise ValueError("times must contain at least one event.")

    if marks.size != times.size:
        raise ValueError("marks must have the same length as times.")

    if np.any(~np.isfinite(times)) or np.any(~np.isfinite(marks)):
        raise ValueError("times and marks must contain finite values only.")

    if np.any(times < 0):
        raise ValueError("times must be non-negative.")

    order = np.argsort(times)
    return times[order], marks[order]


@dataclass
class UnivariateMarkTimeScaledHawkesExpMLE:
    """Univariate exponential Hawkes with mark-dependent time scale.

    Model convention
    ----------------
    lambda(t) = mu + sum_{tj < t}
                alpha * c_j * exp(-beta * (t - tj) / m_j)

    where m_j is a positive normalized mark.

    If normalize_kernel_mass=False:

        c_j = 1

    so:

        phi(u, m_j) = alpha * exp(-beta * u / m_j)

    This is exactly the user's proposed specification.

    If normalize_kernel_mass=True:

        c_j = 1 / m_j

    so:

        phi(u, m_j) = (alpha / m_j) * exp(-beta * u / m_j)

    This makes the total kernel mass independent of m_j.

    Parameters estimated
    --------------------
    mu > 0, alpha >= 0, beta > 0.

    No extra mark parameter is estimated.

    Stability condition
    -------------------
    For normalize_kernel_mass=False:

        branching_ratio = (alpha / beta) * mean(m_j)

    For normalize_kernel_mass=True:

        branching_ratio = alpha / beta

    if m_j is positive.
    """

    T: Optional[float] = None
    enforce_stationarity: bool = True
    normalize_kernel_mass: bool = False
    mark_transform: str = "log1p_ratio"
    epsilon: float = 1e-9

    result_: Optional[object] = None
    params_: Optional[Dict[str, float]] = None
    loglik_: Optional[float] = None
    mark_stats_: Optional[Dict[str, float]] = None

    @staticmethod
    def _logit(p: float) -> float:
        p = np.clip(p, 1e-6, 1.0 - 1e-6)
        return float(np.log(p / (1.0 - p)))

    @staticmethod
    def _sigmoid(x: float) -> float:
        if x >= 0:
            z = np.exp(-x)
            return float(1.0 / (1.0 + z))
        z = np.exp(x)
        return float(z / (1.0 + z))

    def _fit_mark_scale(self, marks: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
        """Convert raw marks to positive dimensionless time-scale factors m_j."""
        marks = np.asarray(marks, dtype=float).ravel()

        if self.mark_transform == "identity":
            x = marks.copy()
            if np.any(x <= 0):
                raise ValueError("With mark_transform='identity', all marks must be > 0.")
            denom = 1.0

        elif self.mark_transform == "ratio_mean":
            x = marks.copy()
            if np.any(x <= 0):
                raise ValueError("With mark_transform='ratio_mean', all marks must be > 0.")
            denom = float(np.mean(x))

        elif self.mark_transform == "log1p_ratio":
            if np.any(marks <= -1):
                raise ValueError("With mark_transform='log1p_ratio', marks must be > -1.")
            x = np.log1p(marks)
            if np.any(x <= 0):
                raise ValueError("log1p(marks) must be positive.")
            denom = float(np.mean(x))

        else:
            raise ValueError(
                "mark_transform must be 'identity', 'ratio_mean', or 'log1p_ratio'."
            )

        denom = max(denom, self.epsilon)
        m = x / denom

        m = np.maximum(m, self.epsilon)

        stats = {
            "denom": denom,
            "mean_m": float(np.mean(m)),
            "min_m": float(np.min(m)),
            "max_m": float(np.max(m)),
        }

        return m, stats

    def _apply_mark_scale(self, marks: np.ndarray) -> np.ndarray:
        """Apply fitted mark scaling to new marks."""
        if self.mark_stats_ is None:
            raise RuntimeError("fit must be called before applying mark scale.")

        marks = np.asarray(marks, dtype=float).ravel()
        denom = self.mark_stats_["denom"]

        if self.mark_transform == "identity":
            x = marks.copy()
            if np.any(x <= 0):
                raise ValueError("With mark_transform='identity', all marks must be > 0.")

        elif self.mark_transform == "ratio_mean":
            x = marks.copy()
            if np.any(x <= 0):
                raise ValueError("With mark_transform='ratio_mean', all marks must be > 0.")

        elif self.mark_transform == "log1p_ratio":
            if np.any(marks <= -1):
                raise ValueError("With mark_transform='log1p_ratio', marks must be > -1.")
            x = np.log1p(marks)

        else:
            raise ValueError(
                "mark_transform must be 'identity', 'ratio_mean', or 'log1p_ratio'."
            )

        m = x / max(denom, self.epsilon)
        return np.maximum(m, self.epsilon)

    def _kernel_multiplier(self, m: np.ndarray) -> np.ndarray:
        """Return c_j."""
        if self.normalize_kernel_mass:
            return 1.0 / np.maximum(m, self.epsilon)
        return np.ones_like(m, dtype=float)

    @staticmethod
    def _A_exact(times: np.ndarray, m: np.ndarray, c: np.ndarray, beta: float) -> np.ndarray:
        """Compute A_i = sum_{j<i} c_j exp(-beta (t_i - t_j) / m_j).

        Exact O(n^2) computation because each event has its own decay beta / m_j.
        """
        n = len(times)
        A = np.zeros(n, dtype=float)

        for i in range(n):
            mask = times[:i] < times[i]
            if np.any(mask):
                u = times[i] - times[:i][mask]
                A[i] = np.sum(c[:i][mask] * np.exp(-beta * u / m[:i][mask]))

        return A

    @classmethod
    def loglikelihood_from_scaled_marks(
        cls,
        times: np.ndarray,
        m: np.ndarray,
        T: float,
        mu: float,
        alpha: float,
        beta: float,
        normalize_kernel_mass: bool = False,
        enforce_stationarity: bool = True,
        epsilon: float = 1e-9,
    ) -> float:
        """Evaluate log-likelihood using already scaled positive marks m_j."""
        times = np.asarray(times, dtype=float).ravel()
        m = np.asarray(m, dtype=float).ravel()

        if T < times[-1]:
            raise ValueError("T must be >= last event time.")

        if mu <= 0 or alpha < 0 or beta <= 0:
            return -np.inf

        if np.any(m <= 0) or np.any(~np.isfinite(m)):
            return -np.inf

        c = 1.0 / m if normalize_kernel_mass else np.ones_like(m)

        average_kernel_mass_factor = float(np.mean(c * m))
        branching_ratio = (alpha / beta) * average_kernel_mass_factor

        if enforce_stationarity and branching_ratio >= 1.0:
            return -np.inf

        A = cls._A_exact(times=times, m=m, c=c, beta=beta)

        intensities = mu + alpha * A

        if np.any(intensities <= 0) or np.any(~np.isfinite(intensities)):
            return -np.inf

        remaining = T - times

        compensator = (
            mu * T
            + alpha
            * np.sum(
                c
                * (m / beta)
                * (1.0 - np.exp(-beta * remaining / m))
            )
        )

        return float(np.sum(np.log(intensities)) - compensator)

    def fit(
        self,
        times: ArrayLike,
        marks: ArrayLike,
        initial: Optional[Tuple[float, float, float]] = None,
    ) -> "UnivariateMarkTimeScaledHawkesExpMLE":
        """Fit mu, alpha, beta by numerical maximization of log-likelihood."""
        times, marks = _sort_times_and_marks(times, marks)

        T = float(times[-1] if self.T is None else self.T)

        if T < times[-1]:
            raise ValueError("T must be >= last event time.")

        n = len(times)

        m, stats = self._fit_mark_scale(marks)
        self.mark_stats_ = stats

        c = self._kernel_multiplier(m)
        average_kernel_mass_factor = float(np.mean(c * m))

        if initial is None:
            empirical_rate = n / T
            mu0 = max(0.5 * empirical_rate, self.epsilon)

            inter_times = np.diff(np.r_[0.0, times])
            beta0 = 1.0 / max(float(np.median(inter_times)), self.epsilon)

            # Conservative initial branching ratio around 0.5
            alpha0 = 0.5 * beta0 / max(average_kernel_mass_factor, self.epsilon)

            initial = (mu0, alpha0, beta0)

        mu0, alpha0, beta0 = map(float, initial)

        if mu0 <= 0 or alpha0 < 0 or beta0 <= 0:
            raise ValueError("initial must satisfy mu>0, alpha>=0, beta>0.")

        if self.enforce_stationarity:
            max_alpha = beta0 / max(average_kernel_mass_factor, self.epsilon)
            if alpha0 >= max_alpha:
                alpha0 = 0.5 * max_alpha

        # Reparameterization:
        #
        # mu = exp(x0)
        # beta = exp(x1)
        #
        # If stationarity is enforced:
        #
        # alpha = beta / average_kernel_mass_factor * sigmoid(x2)
        #
        # This enforces:
        #
        # (alpha / beta) * average_kernel_mass_factor < 1.
        x0 = np.array(
            [
                np.log(mu0),
                np.log(beta0),
                self._logit(
                    alpha0
                    * average_kernel_mass_factor
                    / beta0
                    if beta0 > 0
                    else 0.5
                )
                if self.enforce_stationarity
                else np.log(max(alpha0, self.epsilon)),
            ],
            dtype=float,
        )

        def unpack(x: np.ndarray) -> Tuple[float, float, float]:
            mu = float(np.exp(x[0]))
            beta = float(np.exp(x[1]))

            if self.enforce_stationarity:
                ratio = self._sigmoid(float(x[2]))
                alpha = beta * ratio / max(average_kernel_mass_factor, self.epsilon)
            else:
                alpha = float(np.exp(x[2]))

            return mu, alpha, beta

        def objective(x: np.ndarray) -> float:
            mu, alpha, beta = unpack(x)

            ll = self.loglikelihood_from_scaled_marks(
                times=times,
                m=m,
                T=T,
                mu=mu,
                alpha=alpha,
                beta=beta,
                normalize_kernel_mass=self.normalize_kernel_mass,
                enforce_stationarity=self.enforce_stationarity,
                epsilon=self.epsilon,
            )

            return -ll if np.isfinite(ll) else 1e100

        res = minimize(
            objective,
            x0,
            method="L-BFGS-B",
        )

        mu, alpha, beta = unpack(res.x)

        branching_ratio = (alpha / beta) * average_kernel_mass_factor

        self.result_ = res

        self.params_ = {
            "mu": mu,
            "alpha": alpha,
            "beta": beta,
            "branching_ratio": branching_ratio,
            "average_kernel_mass_factor": average_kernel_mass_factor,
            "normalize_kernel_mass": float(self.normalize_kernel_mass),
            "mark_mean_m": float(np.mean(m)),
            "mark_min_m": float(np.min(m)),
            "mark_max_m": float(np.max(m)),
        }

        self.loglik_ = self.loglikelihood_from_scaled_marks(
            times=times,
            m=m,
            T=T,
            mu=mu,
            alpha=alpha,
            beta=beta,
            normalize_kernel_mass=self.normalize_kernel_mass,
            enforce_stationarity=self.enforce_stationarity,
            epsilon=self.epsilon,
        )

        return self

    def intensity_at_events(self, times: ArrayLike, marks: ArrayLike) -> np.ndarray:
        """Return fitted intensities lambda(t_i-) at observed event times."""
        if self.params_ is None:
            raise RuntimeError("fit must be called before intensity_at_events.")

        times, marks = _sort_times_and_marks(times, marks)

        m = self._apply_mark_scale(marks)
        c = self._kernel_multiplier(m)

        A = self._A_exact(
            times=times,
            m=m,
            c=c,
            beta=self.params_["beta"],
        )

        return self.params_["mu"] + self.params_["alpha"] * A

    def cumulative_intensity(
        self,
        times: ArrayLike,
        marks: ArrayLike,
        eval_times: Optional[ArrayLike] = None,
    ) -> np.ndarray:
        """Compute cumulative intensity Lambda(t) = int_0^t lambda(s) ds."""
        if self.params_ is None:
            raise RuntimeError("fit must be called before cumulative_intensity.")

        times, marks = _sort_times_and_marks(times, marks)

        if eval_times is None:
            eval_times_arr = times.copy()
        else:
            eval_times_arr = np.asarray(eval_times, dtype=float).ravel()

        if np.any(eval_times_arr < 0):
            raise ValueError("eval_times must be non-negative.")

        m = self._apply_mark_scale(marks)
        c = self._kernel_multiplier(m)

        mu = self.params_["mu"]
        alpha = self.params_["alpha"]
        beta = self.params_["beta"]

        out = np.zeros_like(eval_times_arr, dtype=float)

        for r, t in enumerate(eval_times_arr):
            mask = times <= t

            if np.any(mask):
                u = t - times[mask]

                excitation_integral = np.sum(
                    c[mask]
                    * (m[mask] / beta)
                    * (1.0 - np.exp(-beta * u / m[mask]))
                )
            else:
                excitation_integral = 0.0

            out[r] = mu * t + alpha * excitation_integral

        return out

    def compensator(
        self,
        times: ArrayLike,
        marks: ArrayLike,
        T: Optional[float] = None,
    ) -> float:
        """Compute Lambda(T), the compensator on [0, T]."""
        if self.params_ is None:
            raise RuntimeError("fit must be called before compensator.")

        times, marks = _sort_times_and_marks(times, marks)

        T_eval = float(self.T if T is None and self.T is not None else T)

        if T is None and self.T is None:
            T_eval = float(times[-1])

        if T_eval < times[-1]:
            raise ValueError("T must be >= last event time.")

        return float(
            self.cumulative_intensity(
                times=times,
                marks=marks,
                eval_times=np.array([T_eval]),
            )[0]
        )

    def rescaled_interarrival_times(
        self,
        times: ArrayLike,
        marks: ArrayLike,
    ) -> np.ndarray:
        """Return Z_i = Lambda(t_i) - Lambda(t_{i-1})."""
        Lambda_events = self.cumulative_intensity(
            times=times,
            marks=marks,
            eval_times=None,
        )

        return np.diff(np.r_[0.0, Lambda_events])

    def transformed_uniform_residuals(
        self,
        times: ArrayLike,
        marks: ArrayLike,
    ) -> np.ndarray:
        """Return U_i = 1 - exp(-Z_i), expected Uniform(0,1) if model is correct."""
        z = self.rescaled_interarrival_times(times, marks)
        return 1.0 - np.exp(-z)
        
import numpy as np

times = np.array([0.10, 0.30, 0.55, 1.20, 1.70, 2.10, 2.80])
volumes = np.array([100, 250, 80, 1000, 300, 150, 700])

# Version exacte proposée :
# phi(u, m) = alpha * exp(-beta * u / m)
model = UnivariateMarkTimeScaledHawkesExpMLE(
    T=3.0,
    enforce_stationarity=True,
    normalize_kernel_mass=False,
    mark_transform="log1p_ratio",
)

model.fit(times, volumes)

print(model.params_)
print("loglik:", model.loglik_)

lambdas = model.intensity_at_events(times, volumes)
Lambda_T = model.compensator(times, volumes, T=3.0)

print("intensités aux événements:", lambdas)
print("Lambda(T):", Lambda_T)

manual_loglik = np.sum(np.log(lambdas)) - Lambda_T
print("loglik manuelle:", manual_loglik)